In [ ]:
!pip install --force-reinstall numpy==1.26.4 h5py==3.11.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 105.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 118.8 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: h5py
    Found existing installation: h5py 3.15.1
    Uninstalling h5py-3.15.1:
      Successfully uninstalled h5py-3.15.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.35.1 requires nu

In [2]:
import gc
import torch

# Clean Python memory
gc.collect()

# Clean CUDA memory if available
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("✔️ Cache cleared successfully!")


✔️ Cache cleared successfully!


In [3]:
from google.colab import drive
import sys

drive.mount('/content/drive', force_remount=True)
sys.path.append('/content/drive/MyDrive/Models/ArCapModel')


Mounted at /content/drive


In [ ]:
import gc
import torch

torch.cuda.empty_cache()

gc.collect()

print("Memory freed")


Memory freed


In [4]:
import time
import gc
import torch.backends.cudnn as cudnn
import torch.optim
import torch.utils.data
import torchvision.transforms as transforms
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence
from models2 import DecoderWithAttention
from the_datasets2 import *
from utils2 import *
from nltk.translate.bleu_score import corpus_bleu


data_folder = "/content/drive/MyDrive/Models/ArCapModel/FinalDataset"
data_name = 'coco_5_cap_per_img_5_min_word_freq'

emb_dim = 1024
attention_dim = 1024
decoder_dim = 1024
dropout = 0.3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cudnn.benchmark = True

start_epoch = 50
epochs = 60
epochs_since_improvement = 0
batch_size = 50
workers = 1
best_bleu4 = 0
print_freq = 100
checkpoint ="/content/drive/MyDrive/Models/ArCapModel/Checkpoints_best/coco_5_cap_per_img_5_min_word_freq_checkpoint.pth.tar"
def main():
    global best_bleu4, epochs_since_improvement, checkpoint, start_epoch, data_name, word_map

    word_map_file = os.path.join(data_folder, 'WORDMAP_' + data_name + '.json')
    with open(word_map_file, 'r') as j:
        word_map = json.load(j)

    if checkpoint is None:
        decoder = DecoderWithAttention(
            attention_dim=attention_dim,
            embed_dim=emb_dim,
            decoder_dim=decoder_dim,
            vocab_size=len(word_map),
            dropout=dropout
        )
        decoder_optimizer = torch.optim.Adamax(params=filter(lambda p: p.requires_grad, decoder.parameters()))
    else:
        torch.serialization.add_safe_globals([DecoderWithAttention])

        checkpoint = torch.load(checkpoint, map_location=device, weights_only=False)

        start_epoch = checkpoint['epoch'] + 1
        epochs_since_improvement = checkpoint['epochs_since_improvement']
        best_bleu4 = checkpoint['bleu-4']
        decoder = checkpoint['decoder']

        decoder_optimizer = torch.optim.Adamax(params=filter(lambda p: p.requires_grad, decoder.parameters()))
        if 'decoder_optimizer_state_dict' in checkpoint:
            decoder_optimizer.load_state_dict(checkpoint['decoder_optimizer_state_dict'])
            print("Loaded optimizer state from checkpoint.")
        else:
            print("No optimizer state found. Starting optimizer from scratch.")

    decoder = decoder.to(device)

    criterion_ce = nn.CrossEntropyLoss().to(device)
    criterion_dis = nn.MultiLabelMarginLoss().to(device)

    train_loader = torch.utils.data.DataLoader(
        CaptionDataset(data_folder, data_name, 'TRAIN'),
        batch_size=batch_size, shuffle=True, num_workers=workers, pin_memory=True)
    val_loader = torch.utils.data.DataLoader(
        CaptionDataset(data_folder, data_name, 'VAL'),
        batch_size=batch_size, shuffle=True, num_workers=workers, pin_memory=True)

    for epoch in range(start_epoch, epochs):
        if epochs_since_improvement == 15:
            break
        if epochs_since_improvement > 0 and epochs_since_improvement % 5 == 0:
            adjust_learning_rate(decoder_optimizer, 0.5)


        train(train_loader, decoder, criterion_ce, criterion_dis, decoder_optimizer, epoch)
        recent_bleu4 = validate(val_loader, decoder, criterion_ce, criterion_dis)

        is_best = recent_bleu4 > best_bleu4
        best_bleu4 = max(recent_bleu4, best_bleu4)
        if not is_best:
            epochs_since_improvement += 1
            print(f"\nEpochs since last improvement: {epochs_since_improvement}\n")
        else:
            epochs_since_improvement = 0

        save_checkpoint(data_name, epoch, epochs_since_improvement, decoder, decoder_optimizer, recent_bleu4, is_best)

        torch.cuda.empty_cache()
        gc.collect()
        print(f"Memory freed after epoch {epoch}")


def train(train_loader, decoder, criterion_ce, criterion_dis, decoder_optimizer, epoch):
    decoder.train()
    batch_time = AverageMeter()
    data_time = AverageMeter()
    losses = AverageMeter()
    top5accs = AverageMeter()

    start = time.time()
    for i, (imgs, caps, caplens) in enumerate(train_loader):
        data_time.update(time.time() - start)
        imgs = imgs.to(device)
        caps = caps.to(device)
        caplens = caplens.to(device)

        scores, scores_d, caps_sorted, decode_lengths, sort_ind = decoder(imgs, caps, caplens)
        scores_d = scores_d.max(1)[0]
        targets = caps_sorted[:, 1:]
        targets_d = torch.zeros(scores_d.size(0), scores_d.size(1)).to(device)
        targets_d.fill_(-1)

        if isinstance(decode_lengths, torch.Tensor):
            decode_lengths = decode_lengths.tolist()
        for b, L in enumerate(decode_lengths):
            L = int(L)
            if L > 1:
                targets_d[b, :L - 1] = targets[b, :L - 1]

        scores = pack_padded_sequence(scores, decode_lengths, batch_first=True, enforce_sorted=False).data
        targets = pack_padded_sequence(targets, decode_lengths, batch_first=True, enforce_sorted=False).data

        loss_d = criterion_dis(scores_d, targets_d.long())
        loss_g = criterion_ce(scores, targets)
        loss = loss_g + (2 * loss_d)

        decoder_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(filter(lambda p: p.requires_grad, decoder.parameters()), 0.25)
        decoder_optimizer.step()

        top5 = accuracy(scores, targets, 5)
        losses.update(loss.item(), sum(decode_lengths))
        top5accs.update(top5, sum(decode_lengths))
        batch_time.update(time.time() - start)
        start = time.time()

        if i % print_freq == 0:
            print(f'Epoch: [{epoch}][{i}/{len(train_loader)}]\t'
                  f'Batch Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  f'Data Load Time {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  f'Loss {losses.val:.4f} ({losses.avg:.4f})\t'
                  f'Top-5 Accuracy {top5accs.val:.3f} ({top5accs.avg:.3f})')


def validate(val_loader, decoder, criterion_ce, criterion_dis):
    decoder.eval()
    batch_time = AverageMeter()
    losses = AverageMeter()
    top5accs = AverageMeter()
    start = time.time()

    references = []
    hypotheses = []

    with torch.no_grad():
        for i, (imgs, caps, caplens, allcaps) in enumerate(val_loader):
            imgs = imgs.to(device)
            caps = caps.to(device)
            caplens = caplens.to(device)

            scores, scores_d, caps_sorted, decode_lengths, sort_ind = decoder(imgs, caps, caplens)
            scores_d = scores_d.max(1)[0]
            targets = caps_sorted[:, 1:]
            targets_d = torch.zeros(scores_d.size(0), scores_d.size(1)).to(device)
            targets_d.fill_(-1)

            if isinstance(decode_lengths, torch.Tensor):
                decode_lengths = decode_lengths.tolist()
            for b, L in enumerate(decode_lengths):
                L = int(L)
                if L > 1:
                    targets_d[b, :L - 1] = targets[b, :L - 1]

            scores_copy = scores.clone()
            packed_scores = pack_padded_sequence(scores, decode_lengths, batch_first=True, enforce_sorted=False)
            packed_targets = pack_padded_sequence(targets, decode_lengths, batch_first=True, enforce_sorted=False)
            scores = packed_scores.data
            targets = packed_targets.data

            loss_d = criterion_dis(scores_d, targets_d.long())
            loss_g = criterion_ce(scores, targets)
            loss = loss_g + (2 * loss_d)

            losses.update(loss.item(), sum(decode_lengths))
            top5 = accuracy(scores, targets, 5)
            top5accs.update(top5, sum(decode_lengths))
            batch_time.update(time.time() - start)
            start = time.time()

            if i % print_freq == 0:
                print(f'Validation: [{i}/{len(val_loader)}]\t'
                      f'Batch Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                      f'Loss {losses.val:.4f} ({losses.avg:.4f})\t'
                      f'Top-5 Accuracy {top5accs.val:.3f} ({top5accs.avg:.3f})')

            allcaps = allcaps[sort_ind.cpu()]
            for j in range(allcaps.shape[0]):
                img_caps = allcaps[j].tolist()
                img_captions = list(map(lambda c: [w for w in c if w not in {word_map['<start>'], word_map['<pad>']}], img_caps))
                references.append(img_captions)

            _, preds = torch.max(scores_copy, dim=2)
            preds = preds.tolist()
            temp_preds = []
            for j, p in enumerate(preds):
                temp_preds.append(preds[j][:decode_lengths[j]])
            preds = temp_preds
            hypotheses.extend(preds)
            assert len(references) == len(hypotheses)

    bleu4 = corpus_bleu(references, hypotheses)
    bleu4 = round(bleu4, 4)

    print(f'\n * LOSS - {losses.avg:.3f}, TOP-5 ACCURACY - {top5accs.avg:.3f}, BLEU-4 - {bleu4}\n')
    return bleu4


if __name__ == '__main__':
    main()


Loaded optimizer state from checkpoint.
Epoch: [50][0/7452]	Batch Time 76.577 (76.577)	Data Load Time 74.891 (74.891)	Loss 3.8101 (3.8101)	Top-5 Accuracy 78.821 (78.821)
Epoch: [50][100/7452]	Batch Time 0.431 (6.123)	Data Load Time 0.000 (5.713)	Loss 3.7150 (3.6909)	Top-5 Accuracy 75.322 (77.248)
Epoch: [50][200/7452]	Batch Time 0.356 (3.542)	Data Load Time 0.000 (3.152)	Loss 3.3477 (3.7330)	Top-5 Accuracy 78.099 (77.005)
Epoch: [50][300/7452]	Batch Time 0.851 (2.570)	Data Load Time 0.000 (2.187)	Loss 4.2349 (3.7098)	Top-5 Accuracy 72.320 (76.988)
Epoch: [50][400/7452]	Batch Time 0.353 (2.057)	Data Load Time 0.000 (1.674)	Loss 3.5460 (3.6943)	Top-5 Accuracy 76.445 (77.035)
Epoch: [50][500/7452]	Batch Time 0.382 (1.738)	Data Load Time 0.000 (1.357)	Loss 3.1896 (3.6946)	Top-5 Accuracy 79.789 (77.003)
Epoch: [50][600/7452]	Batch Time 0.296 (1.516)	Data Load Time 0.000 (1.138)	Loss 4.1656 (3.7052)	Top-5 Accuracy 77.234 (76.980)
Epoch: [50][700/7452]	Batch Time 0.338 (1.360)	Data Load Time 